# Age vs Life Expectancy

People usually quote **life expectancy at birth** — about 80–85 years in developed countries. But life expectancy is really a function of *current age*: once you have already survived to age $x$, your **remaining** life expectancy $e_x$ is the expected number of additional years of life.

This notebook builds a period life table that mimics the **mortality structure of developed countries** (very low childhood/adult mortality, then a Gompertz rise at older ages) and plots age against remaining life expectancy.

## Life-Table Idea

Given age-specific mortality $\mu(x)$, a life table tracks a synthetic cohort of $l_0$ births:

- $l_x$ — survivors to exact age $x$
- $L_x$ — person-years lived between $x$ and $x+1$
- $T_x = \sum_{y \ge x} L_y$ — total person-years remaining after age $x$
- $e_x = T_x / l_x$ — remaining life expectancy at age $x$

The expected age at death for someone currently age $x$ is then $x + e_x$, which **increases** with $x$ (survivors are selected for longer lives).

In [ ]:
import math
from dataclasses import dataclass

import numpy as np


# Developed-country style mortality (Gompertz-Makeham + infant excess).
# Calibrated so e_0 is roughly 81-83 years — typical of OECD / "more developed" regions.
MAKEHAM_A = 0.00025         # background accident/disease floor
GOMPERTZ_B = 2.0e-5         # level of senescent mortality
GOMPERTZ_C = 0.094          # rate of aging
INFANT_EXTRA_Q0 = 0.003     # extra first-year mortality (developed-country low)
CHILD_EXTRA_PEAK = 0.0002   # tiny residual child mortality bump
MAX_AGE = 110
RADIX = 100_000


def force_of_mortality(age: float) -> float:
    """Continuous hazard mu(x) for adult ages (Gompertz-Makeham)."""
    return MAKEHAM_A + GOMPERTZ_B * math.exp(GOMPERTZ_C * age)


def qx_from_mu(age: int) -> float:
    """Probability of dying between age and age+1, with early-age adjustments."""
    if age == 0:
        # Constant-force approximation plus developed-country infant excess
        mu = force_of_mortality(0.5)
        q = 1.0 - math.exp(-mu) + INFANT_EXTRA_Q0
    elif age < 15:
        mu = force_of_mortality(age + 0.5)
        # Slightly elevated relative to pure Gompertz at very young ages, then fading
        child_extra = CHILD_EXTRA_PEAK * max(0.0, 1.0 - age / 15.0)
        q = 1.0 - math.exp(-(mu + child_extra))
    else:
        mu = force_of_mortality(age + 0.5)
        q = 1.0 - math.exp(-mu)
    return float(min(max(q, 0.0), 1.0))


@dataclass
class LifeTable:
    ages: np.ndarray
    qx: np.ndarray
    lx: np.ndarray
    Lx: np.ndarray
    Tx: np.ndarray
    ex: np.ndarray


def build_life_table(max_age: int = MAX_AGE, radix: int = RADIX) -> LifeTable:
    ages = np.arange(0, max_age + 1)
    qx = np.array([qx_from_mu(int(a)) for a in ages], dtype=float)
    qx[-1] = 1.0  # close out the table

    lx = np.zeros_like(qx)
    lx[0] = float(radix)
    for i in range(len(ages) - 1):
        lx[i + 1] = lx[i] * (1.0 - qx[i])

    # Person-years: average of survivors in the year (uniform deaths assumption)
    Lx = np.zeros_like(qx)
    for i in range(len(ages) - 1):
        Lx[i] = (lx[i] + lx[i + 1]) / 2.0
    Lx[-1] = lx[-1] / max(force_of_mortality(max_age), 1e-6)  # residual lifetime

    Tx = np.cumsum(Lx[::-1])[::-1]
    ex = np.divide(Tx, lx, out=np.zeros_like(Tx), where=lx > 0)

    return LifeTable(ages=ages, qx=qx, lx=lx, Lx=Lx, Tx=Tx, ex=ex)


lt = build_life_table()

print(f"Life expectancy at birth e_0: {lt.ex[0]:.2f} years")
print(f"Remaining LE at age 40:       {lt.ex[40]:.2f} years  (expected age at death {40 + lt.ex[40]:.1f})")
print(f"Remaining LE at age 65:       {lt.ex[65]:.2f} years  (expected age at death {65 + lt.ex[65]:.1f})")
print(f"Remaining LE at age 80:       {lt.ex[80]:.2f} years  (expected age at death {80 + lt.ex[80]:.1f})")
print(f"Survivors to age 65 (of {RADIX:,}): {lt.lx[65]:,.0f}  ({100 * lt.lx[65] / RADIX:.1f}%)")

## Notes and References

- **Developed-country mortality structure**: near-rectangular survivorship — most people reach old age; deaths concentrate after ~70. That is the demographic counterpart of high $e_0$.
- **Gompertz–Makeham law**: adult mortality rises roughly exponentially with age, which is a good first-order description for modern low-mortality populations.
- Parameters here are illustrative, tuned so $e_0 \approx 82$ years — in the ballpark of UN "More developed regions" / OECD averages — not a fit to one specific country-year.
- See [Life table](https://en.wikipedia.org/wiki/Life_table) and [Life expectancy](https://en.wikipedia.org/wiki/Life_expectancy) for background.
- Period life tables describe a synthetic cohort living through today's age-specific rates; they are not a forecast of any real birth cohort.

## Create and Display the Plot

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        plt.style.use('default')
        print("Using default matplotlib style")

ages = lt.ages
remaining = lt.ex
expected_death_age = ages + remaining

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# --- Left: remaining life expectancy vs age ---
ax = axes[0]
ax.plot(ages, remaining, linewidth=2.5, color='#007AFF', label='Remaining life expectancy $e_x$')

# Reference: naive "years left if you die at e_0"
naive = np.maximum(lt.ex[0] - ages, 0)
ax.plot(ages, naive, linewidth=1.8, color='#FF6B6B', linestyle='--', alpha=0.85,
        label=r'Naive: $e_0 - x$ (cut off at 0)')

markers = {
    0: 'At birth',
    40: 'Age 40',
    65: 'Age 65',
    80: 'Age 80',
}
for age, label in markers.items():
    ax.plot(age, remaining[age], 'o', markersize=10, color='#FF6B6B', zorder=5)
    ax.annotate(
        f'{label}\n$e_{{{age}}}$ = {remaining[age]:.1f} y',
        xy=(age, remaining[age]),
        xytext=(12, 12 if age < 70 else -28),
        textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.45', facecolor='white',
                  edgecolor='gray', alpha=0.9, linewidth=1.2),
        fontsize=9,
        ha='left',
    )

ax.set_xlabel('Current age (years)', fontsize=12)
ax.set_ylabel('Remaining life expectancy (years)', fontsize=12)
ax.set_title('Remaining Life Expectancy vs Age\n(Developed-country mortality structure)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, 100)
ax.set_ylim(0, 90)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.25)

# --- Right: expected age at death ---
ax2 = axes[1]
ax2.plot(ages, expected_death_age, linewidth=2.5, color='#34C759',
         label=r'Expected age at death ($x + e_x$)')
ax2.axhline(lt.ex[0], color='#FF6B6B', linestyle='--', linewidth=1.5, alpha=0.8,
            label=fr'Life expectancy at birth $e_0$ = {lt.ex[0]:.1f}')
ax2.plot(ages, ages, color='gray', linestyle=':', linewidth=1.2, alpha=0.7, label=r'Current age ($x$)')

for age in (0, 40, 65, 80):
    ax2.plot(age, expected_death_age[age], 'o', markersize=10, color='#007AFF', zorder=5)
    ax2.annotate(
        f'{expected_death_age[age]:.1f}',
        xy=(age, expected_death_age[age]),
        xytext=(8, 8),
        textcoords='offset points',
        fontsize=9,
    )

ax2.set_xlabel('Current age (years)', fontsize=12)
ax2.set_ylabel('Expected age at death (years)', fontsize=12)
ax2.set_title('Expected Age at Death Rises with Age\n(survivorship selection)',
              fontsize=13, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 110)
ax2.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax2.grid(True, alpha=0.25)

source_text = (
    'Model assumptions:\n'
    '• Gompertz-Makeham adult mortality\n'
    '• Low infant/child mortality (developed)\n'
    f'• Calibrated to e_0 ≈ {lt.ex[0]:.0f} years\n'
    '• Period life table (synthetic cohort)'
)
fig.text(0.5, 0.02, source_text, ha='center', fontsize=9,
         family='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#f5f5f5', edgecolor='gray', alpha=0.95))

fig.tight_layout(rect=[0, 0.12, 1, 1])

out_path = Path('age_life_expectancy.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved plot to {out_path.resolve()}')
plt.show()

## Takeaways

1. **Remaining life expectancy falls with age**, but *slower* than the naive $e_0 - x$ line — because the life table already conditions on having survived to age $x$.
2. **Expected age at death ($x + e_x$) rises with age.** At birth it equals $e_0$; by age 80 it is typically well above $e_0$. Surviving is informative.
3. **Developed-country structure** shows up as a high $e_0$ and a large share of the cohort still alive at 65+, which is why the right-hand curve stays high deep into old age.